# RMSNorm: From Scratch

**Root Mean Square Normalization (RMSNorm)** is a simplification of LayerNorm. It was popularized by models like LLaMA and Gopher.

In this tutorial, we will:
1. Understand the math behind Normalization.
2. Implement RMSNorm from scratch using raw PyTorch operations.
3. Compare it with LayerNorm.

## 1. The Math

Standard LayerNorm centers the data (subtracts mean) and scales it (divides by variance).

$$ \text{LayerNorm}(x) = \frac{x - \mu}{\sigma} * \gamma + \beta $$

RMSNorm assumes that the mean is close to 0, so we can skip the centering step. We only scale by the Root Mean Square (RMS).

$$ \text{RMS}(x) = \sqrt{\frac{1}{n} \sum_{i=1}^n x_i^2 + \epsilon} $$

$$ \text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} * \gamma $$

Where $\gamma$ is a learnable scale parameter.

## 2. Implementation from Scratch

Let's implement this step-by-step.

In [ ]:
import torch
import torch.nn as nn

class MyRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # The learnable parameter gamma (scale)
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x shape: [batch, seq_len, dim]
        
        # 1. Calculate sum of squares
        # We want to normalize over the last dimension (dim)
        sum_squares = torch.pow(x, 2).sum(dim=-1, keepdim=True)
        
        # 2. Calculate mean of squares
        input_dtype = x.dtype
        n = x.shape[-1]
        mean_squares = sum_squares / n
        
        # 3. Calculate RMS (add epsilon for stability)
        rms = torch.sqrt(mean_squares + self.eps)
        
        # 4. Normalize
        x_norm = x / rms
        
        # 5. Scale by gamma
        return x_norm * self.weight

# Test it
x = torch.randn(2, 5)
my_norm = MyRMSNorm(5)
output = my_norm(x)

print("Input:", x)
print("Output:", output)

# Verify that output has roughly unit norm
print("Output RMS:", torch.sqrt(output.pow(2).mean(dim=-1)))

## 3. Comparison with PyTorch

Let's verify our implementation against the optimized PyTorch version.

In [ ]:
torch_norm = nn.RMSNorm(5)
# Copy weights to ensure fair comparison
torch_norm.weight.data = my_norm.weight.data

output_torch = torch_norm(x)

print("Difference:", (output - output_torch).abs().max().item())